# XAI Enhancer Module - Colab Runner

This notebook allows you to run the XAI Enhancer module on Google Colab. It handles environment setup, sample data creation (using Hugging Face Datasets), and model downloading.

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install torch torchvision pillow numpy pandas tqdm grad-cam datasets huggingface_hub

# Confirm we are in the correct directory
import os
print(f"Current working directory: {os.getcwd()}")
if os.path.basename(os.getcwd()) != "XAI_Enhancer_module":
    print("⚠️ Warning: Please make sure you are in the 'XAI_Enhancer_module' directory.")
    if os.path.exists("XAI_Enhancer_module"):
        os.chdir("XAI_Enhancer_module")
        print(f"Changed directory to: {os.getcwd()}")

## 2. Create Sample Dataset (5000 Images)

We will stream the first 5000 images from the **ImageNet-1k validation set** using Hugging Face Datasets.

**Important**: ImageNet-1k is a gated dataset. You need to:
1.  Have a Hugging Face account.
2.  Accept terms at [https://huggingface.co/datasets/imagenet-1k](https://huggingface.co/datasets/imagenet-1k).
3.  Enter your Access Token below.

In [ ]:
from datasets import load_dataset
from huggingface_hub import login
from pathlib import Path
from tqdm import tqdm
import json
import urllib.request

# --- 1. Login to Hugging Face ---
# This will prompt for your token if not already logged in
print("Please enter your Hugging Face Access Token when prompted (or if a widget appears):")
login(add_to_git_credential=False)

def create_imagenet_sample_from_hf(target_count=5000, base_path="imagenet_val_sample"):
    print(f"\nStarting download of {target_count} images from ImageNet-1k validation set...")
    
    # Load streaming dataset so we don't finish disk space
    try:
        ds = load_dataset("imagenet-1k", split="validation", streaming=True, trust_remote_code=True)
    except Exception as e:
        print(f"\n❌ Error loading dataset: {e}")
        print("did you accept the manual terms at https://huggingface.co/datasets/imagenet-1k ?")
        return None

    # --- Synset Mapping Handling ---
    # The evaluation scripts rely on LOC_synset_mapping.txt being present in the parent directory.
    mapping_path = None
    possible_paths = [
        "../LOC_synset_mapping.txt",
        "./LOC_synset_mapping.txt",
        "LOC_synset_mapping.txt"
    ]
    
    for p in possible_paths:
        if Path(p).exists():
            mapping_path = Path(p)
            print(f"Found synset mapping at: {mapping_path}")
            break

    # Always download the official class index JSON for our own lookup
    url = "https://s3.amazonaws.com/deep-learning-models/image-models/imagenet_class_index.json"
    try:
        with urllib.request.urlopen(url) as response:
            class_index = json.load(response)
        # Format example: "0": ["n01440764", "tench"]
    except Exception as e:
        print(f"Could not download class index: {e}")
        return None

    # If text file missing, re-create it from the JSON so imports work
    if not mapping_path:
        mapping_path = Path("../LOC_synset_mapping.txt")
        print(f"⚠️ Synset mapping file missing. Generating it at {mapping_path}...")
        try:
            with open(mapping_path, 'w') as f:
                for idx in range(1000):
                    # class_index indices are strings "0", "1"...
                    entry = class_index[str(idx)]
                    synset = entry[0]
                    name = entry[1]
                    # Format: nXXXXXX class_name
                    f.write(f"{synset} {name}\n")
            print("✅ Created synset mapping file.")
        except Exception as e:
            print(f"Failed to write mapping file: {e}")

    base_dir = Path(base_path)
    base_dir.mkdir(parents=True, exist_ok=True)
    
    count = 0
    print("Streaming and saving images...")
    
    for sample in tqdm(ds, total=target_count):
        if count >= target_count:
            break
        
        img = sample['image']
        label_idx = sample['label'] # Integer 0-999
        
        # Get synset ID
        synset_id = class_index[str(label_idx)][0]
        
        # Create folder
        synset_dir = base_dir / synset_id
        synset_dir.mkdir(exist_ok=True)
        
        # Save image
        if img.mode != 'RGB':
            img = img.convert('RGB')
            
        save_path = synset_dir / f"val_{count}.JPEG"
        if not save_path.exists():
            img.save(save_path)
        
        count += 1
        
    print(f"\n✅ Successfully saved {count} images to {base_path}")
    return str(base_dir)

# Run the creation
dataset_path = create_imagenet_sample_from_hf(target_count=5000)
if dataset_path:
    print(f"Dataset ready at: {dataset_path}")
else:
    print("Failed to create dataset.")

## 3. Download Models

We need to create a local directory for models so the script can find them.

In [ ]:
from download_models import download_all_models

# Set a local cache directory
MODEL_CACHE_DIR = "./pytorch_models"

# Download models (this might take a few minutes)
download_all_models(custom_folder=MODEL_CACHE_DIR)

## 4. Run Evaluation

Now we can run the `imagenet_evaluation.py` script using our 5000-image dataset and cached models.

In [ ]:
if dataset_path:
    # Run comparison evaluation
    # Note: Analyzing 5000 images will take significant time on CPU. 
    # Ensure you are using a GPU Runtime (Runtime -> Change runtime type -> T4 GPU).
    !python imagenet_evaluation.py \
      --model resnet50 \
      --imagenet-path {dataset_path} \
      --eval-type comparison \
      --max-images 5000 \
      --model-cache-dir ./pytorch_models \
      --device auto \
      --layer-mode last \
      --methods GradCAM HiResCAM
else:
    print("Skipping evaluation as dataset was not created.")

## 5. View Results

Results are saved in `csv_exports` and `analysis_results`.

In [ ]:
import pandas as pd
import glob

# Find the latest CSV result
csv_files = glob.glob("csv_exports/*/*.csv")
if csv_files:
    latest_csv = max(csv_files, key=os.path.getctime)
    print(f"Reading results from: {latest_csv}")
    df = pd.read_csv(latest_csv)
    display(df)
else:
    print("No CSV results found yet.")